In [1]:
import pandas as pd

import spacy
import stanza
from tqdm.auto import tqdm
import nltk
from nltk.corpus import stopwords

tqdm.pandas()

c:\Users\USER\anaconda3\envs\IR\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
INPUT_CSV = (
    "Greek_Parliament_Proceedings_1989_2020/Greek_Parliament_Proceedings_1989_2020.csv"
)

OUTPUT_CSV = "dataset/greek_parliament_preprocessed.csv"

In [3]:
dataframe = pd.read_csv(INPUT_CSV)

In [4]:
dropped_columns = [
    "parliamentary_period",
    "parliamentary_session",
    "parliamentary_sitting",
    "member_region",
    "government",
    "roles",
]
dataframe.drop(columns=dropped_columns, inplace=True)
dataframe.dropna(inplace=True)

Preprocessing Greek Parliament Proceedings Dataset
===========================================

In [5]:
preprocess = True

Remove stopwords
------------------

In [6]:
if preprocess:

    def remove_spaces(text):
        return " ".join(text.split())

    def remove_accents(text):
        return text.translate(str.maketrans("άέήίόύώϊϋΐΰ", "αεηιουωιυιυ"))

    nltk.download("stopwords")
    greek_stopwords = set(stopwords.words("greek"))

    def remove_stopwords_nltk(text):
        tokens = text.split()
        tokens = [word for word in tokens if word not in greek_stopwords]
        return " ".join(tokens)

    # stanza.download("el")
    # stanza_nlp = stanza.Pipeline(
    #     "el", processors="tokenize,mwt,pos,lemma", use_gpu=True
    # )

    # def preprocess_with_stanza(text):
    #     # Use Stanza to process the text
    #     doc = stanza_nlp(text)

    #     tokens = [
    #         word.lemma.lower()
    #         for sentence in doc.sentences
    #         for word in sentence.words
    #         if word.upos not in ["PUNCT", "SYM", "NUM"]
    #     ]
    #     return " ".join(tokens)

    # spacy_nlp = spacy.load("el_core_news_lg")

    # def remove_stopwords_spacy(text):
    #     doc = spacy_nlp(text)
    #     tokens = [token.lemma_ for token in doc if not token.is_stop and token.is_alpha]
    #     return " ".join(tokens)

    # def preprocess_with_spacy(text):
    #     doc = spacy_nlp(text)
    #     tokens = [token.lemma_ for token in doc if not token.is_stop and token.is_alpha]
    #     return " ".join(tokens)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [7]:
if preprocess:
    dataframe.speech = dataframe.speech.progress_apply(remove_spaces)
    dataframe.speech = dataframe.speech.progress_apply(remove_accents)
    dataframe.speech = dataframe.speech.progress_apply(remove_stopwords_nltk)
    # dataframe["preprocessed"] = dataframe.speech.progress_apply(preprocess_with_spacy)

100%|██████████| 1233057/1233057 [00:28<00:00, 43711.79it/s]


In [9]:
# filter rows with empty speeches
dataframe.dropna(inplace=True)
dataframe = dataframe[(dataframe["speech"].str.strip() != "")]

In [10]:
# save the preprocessed dataframe to a new CSV file
dataframe.to_csv(OUTPUT_CSV, index=False)

In [11]:
dataframe

,member_name,sitting_date,political_party,member_gender,speech
0,κρητικος νικολαου παναγιωτης,03/07/1989,πανελληνιο σοσιαλιστικο κινημα,male,Παρακαλειται Γραμματεας κ. Βουλγαρακης συνοδευ...
1,κρητικος νικολαου παναγιωτης,03/07/1989,πανελληνιο σοσιαλιστικο κινημα,male,Παρακαλειται κυριος Γραμματεας συνοδευσει Ιερα...
2,κρητικος νικολαου παναγιωτης,03/07/1989,πανελληνιο σοσιαλιστικο κινημα,male,"Κυριοι συναδελφοι, παρακαλω τη Βουλη εξουσιοδο..."
4,κρητικος νικολαου παναγιωτης,03/07/1989,πανελληνιο σοσιαλιστικο κινημα,male,Η Βουλη παρεσχε τη ζητηθεισα εξουσιοδοτηση. Με...
5,κρητικος νικολαου παναγιωτης,04/07/1989,πανελληνιο σοσιαλιστικο κινημα,male,"Υπαρχει κανεις εκ κυριων συναδελφων, οποιος ψη..."
...,...,...,...,...,...
1280911,κωνσταντινοπουλος κωνσταντινου οδυσσεας,24/07/2020,κινημα αλλαγης,male,"Οι θεσεις κομματων, οπως αποτυπωθηκαν ψηφιση η..."
1280912,κωνσταντινοπουλος κωνσταντινου οδυσσεας,24/07/2020,κινημα αλλαγης,male,Μετα ολοκληρωση της ψηφοφοριας ηλεκτρονικο συσ...
1280913,κωνσταντινοπουλος κωνσταντινου οδυσσεας,24/07/2020,κινημα αλλαγης,male,"Κυριες κυριοι συναδελφοι, παρακαλω Σωμα εξουσι..."
1280915,κωνσταντινοπουλος κωνσταντινου οδυσσεας,24/07/2020,κινημα αλλαγης,male,Το Σωμα παρεσχε τη ζητηθεισα εξουσιοδοτηση.Κυρ...
